# Image Classification

## Learning Objectives
1. Implement 2D convolution from scratch in numpy
2. Build and train a CNN with residual connections in PyTorch
3. Compare training-from-scratch vs transfer learning vs feature extraction
4. Implement a manual data augmentation pipeline without torchvision


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: 2D Convolution from Scratch (numpy)

In [ ]:
# ---- Level 1: Manual conv2d in numpy -------------------------------------
# A single convolution applies a learned filter by sliding it over the input.
# Understanding the manual computation clarifies what torch.nn.Conv2d does.

def conv2d_numpy(image, kernel, stride=1, padding=0):
    """Apply a 2D convolution kernel to a 2D image.

    Args:
        image: 2D numpy array of shape (H, W)
        kernel: 2D numpy array of shape (kH, kW)
        stride: step size for sliding window
        padding: zero-padding added to each side

    Returns:
        Feature map of shape (out_H, out_W)
    """
    H, W = image.shape
    kH, kW = kernel.shape
    # Pad input with zeros on all sides
    if padding > 0:
        image = np.pad(image, padding, mode='constant')
    out_H = (H + 2 * padding - kH) // stride + 1
    out_W = (W + 2 * padding - kW) // stride + 1
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            row = i * stride
            col = j * stride
            patch = image[row:row + kH, col:col + kW]
            # Element-wise multiply and sum = one convolution step
            output[i, j] = np.sum(patch * kernel)
    return output


def relu_numpy(x):
    """Element-wise ReLU: max(0, x)."""
    return np.maximum(0, x)


def maxpool2d_numpy(feature_map, pool_size=2, stride=2):
    """2x2 max pooling — halves spatial dimensions."""
    H, W = feature_map.shape
    out_H = (H - pool_size) // stride + 1
    out_W = (W - pool_size) // stride + 1
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            patch = feature_map[
                i * stride:i * stride + pool_size,
                j * stride:j * stride + pool_size,
            ]
            output[i, j] = np.max(patch)
    return output


# Create a synthetic 8x8 grayscale image with a vertical edge in the middle
image = np.zeros((8, 8))
image[:, 4:] = 1.0  # right half is bright

# Sobel vertical edge detector kernel
sobel_v = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=float)

# Apply conv + relu + pool
feature_map = conv2d_numpy(image, sobel_v, stride=1, padding=1)
activated = relu_numpy(feature_map)
pooled = maxpool2d_numpy(activated, pool_size=2, stride=2)

print('Image shape:     ', image.shape)
print('After conv shape:', feature_map.shape)
print('After relu shape:', activated.shape)
print('After pool shape:', pooled.shape)
print('Feature map (vertical edges detected):')
print(np.round(feature_map, 1))

# Visualize the pipeline
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, data, title in zip(
    axes,
    [image, feature_map, activated, pooled],
    ['Input Image', 'Conv Output', 'After ReLU', 'After MaxPool'],
):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.suptitle('Level 1: Manual Conv → ReLU → MaxPool pipeline')
plt.tight_layout()
plt.savefig('/tmp/cv01_level1.png', dpi=80)
plt.close()
print('Saved: /tmp/cv01_level1.png')


## Level 2: CNN with Residual Connections in PyTorch

In [ ]:
# ---- Level 2: CNN + residual connections on synthetic CIFAR-like data ----
# Residual connections are the core of ResNet: they add the layer input
# to the output, creating gradient highways that prevent vanishing gradients.


class ConvBNReLU(nn.Module):
    """Standard conv-batchnorm-relu block used throughout ResNet-style models."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResidualBlock(nn.Module):
    """One residual block: two ConvBNReLU layers with a skip connection.

    If in_ch != out_ch, a 1x1 conv projects the skip to match dimensions.
    """

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = ConvBNReLU(in_ch, out_ch, stride=stride)
        # Second conv does not change spatial size
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, 1, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.act = nn.ReLU(inplace=True)
        # Projection shortcut: align channel dimensions when they change
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        # The skip path adds the input directly to the transformed output
        return self.act(self.conv2(self.conv1(x)) + self.skip(x))


class SmallResNet(nn.Module):
    """Lightweight ResNet-style classifier: 3 residual blocks + global avg pool."""

    def __init__(self, num_classes=10):
        super().__init__()
        # Stem: initial wide conv to extract low-level features
        self.stem = ConvBNReLU(3, 32)
        # Residual stages with increasing channels and downsampling
        self.stage1 = ResidualBlock(32, 64, stride=2)
        self.stage2 = ResidualBlock(64, 128, stride=2)
        self.stage3 = ResidualBlock(128, 256, stride=2)
        # Global average pool collapses spatial dims to 1x1
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.gap(x).flatten(1)  # (B, 256)
        return self.classifier(x)


# --- Generate synthetic CIFAR-like dataset (3 x 32 x 32, 10 classes) --------
N_TRAIN = 1000
N_VAL = 200
N_CLASSES = 10
IMG_SIZE = 32

# Each class has a distinct mean color so the network can solve the task
X_all = torch.zeros(N_TRAIN + N_VAL, 3, IMG_SIZE, IMG_SIZE)
y_all = torch.zeros(N_TRAIN + N_VAL, dtype=torch.long)
for cls in range(N_CLASSES):
    n = (N_TRAIN + N_VAL) // N_CLASSES
    start = cls * n
    # Class-specific color signal + noise for synthetic classification task
    X_all[start:start + n] = (
        torch.randn(n, 3, IMG_SIZE, IMG_SIZE) * 0.3
        + torch.tensor([cls / N_CLASSES, cls / (2 * N_CLASSES), 0.5])
        .view(1, 3, 1, 1)
    )
    y_all[start:start + n] = cls

# Normalize to zero mean, unit variance (simulates ImageNet preprocessing)
mean = X_all.mean(dim=(0, 2, 3), keepdim=True)
std = X_all.std(dim=(0, 2, 3), keepdim=True) + 1e-7
X_all = (X_all - mean) / std

dataset = TensorDataset(X_all, y_all)
train_ds, val_ds = random_split(dataset, [N_TRAIN, N_VAL])
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# --- Training loop ---------------------------------------------------------
model = SmallResNet(num_classes=N_CLASSES).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-4)
# Cosine annealing gradually reduces LR to near zero — improves final accuracy
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
criterion = nn.CrossEntropyLoss()

train_losses, val_accuracies = [], []
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        try:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                print('OOM: reduce batch_size or model width')
            else:
                raise
    scheduler.step()
    # Validate every 5 epochs to monitor overfitting
    if (epoch + 1) % 5 == 0:
        model.eval()
        correct = 0
        with torch.no_grad():
            for X_v, y_v in val_loader:
                preds = model(X_v.to(device)).argmax(1)
                correct += (preds == y_v.to(device)).sum().item()
        val_acc = correct / N_VAL
        val_accuracies.append((epoch + 1, val_acc))
    train_losses.append(epoch_loss / len(train_loader))

print('Training complete.')
for ep, acc in val_accuracies:
    print(f'  Epoch {ep:3d}: val_accuracy = {acc:.3f}')

# Count parameters to understand model size
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}  Trainable: {trainable_params:,}')


## Real-World Example 1: Transfer Learning Simulation

In [ ]:
# ---- RW1: Transfer learning — freeze backbone, train head only -----------
# Simulates using a 'pretrained' model: freeze all layers except the final
# classification head, then compare accuracy with full fine-tune.


def freeze_backbone(model):
    """Freeze all parameters except the classifier head.

    Freezing the backbone preserves learned features.
    Only the randomly-initialized head is updated.
    """
    for name, param in model.named_parameters():
        if 'classifier' not in name:
            param.requires_grad = False


def unfreeze_all(model):
    """Unfreeze all parameters for full fine-tuning."""
    for param in model.parameters():
        param.requires_grad = True


def train_eval(model, train_loader, val_loader, lr, epochs, tag):
    """Shared training loop for comparing strategies.

    Returns list of (epoch, val_accuracy) tuples.
    """
    # Only pass parameters with requires_grad=True to optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    opt = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    results = []
    for epoch in range(epochs):
        model.train()
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(model(X_b), y_b)
            loss.backward()
            opt.step()
        # Eval
        model.eval()
        correct = 0
        with torch.no_grad():
            for X_v, y_v in val_loader:
                preds = model(X_v.to(device)).argmax(1)
                correct += (preds == y_v.to(device)).sum().item()
        val_acc = correct / N_VAL
        results.append(val_acc)
    print(f'{tag}: final val_acc = {results[-1]:.3f}')
    return results


# Strategy 1: Head-only training (feature extraction)
model_fe = SmallResNet(num_classes=N_CLASSES).to(device)
freeze_backbone(model_fe)
n_trainable_fe = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
print(f'Feature extraction: {n_trainable_fe:,} trainable params')
results_fe = train_eval(model_fe, train_loader, val_loader, lr=1e-3, epochs=15,
                        tag='Feature Extraction')

# Strategy 2: Full fine-tune (all layers updated)
model_ft = SmallResNet(num_classes=N_CLASSES).to(device)
unfreeze_all(model_ft)
n_trainable_ft = sum(p.numel() for p in model_ft.parameters() if p.requires_grad)
print(f'Full fine-tune:     {n_trainable_ft:,} trainable params')
results_ft = train_eval(model_ft, train_loader, val_loader, lr=1e-4, epochs=15,
                        tag='Full Fine-tune')

# Plot comparison
plt.figure(figsize=(8, 4))
plt.plot(range(1, 16), results_fe, label='Feature Extraction (frozen backbone)')
plt.plot(range(1, 16), results_ft, label='Full Fine-tune')
plt.xlabel('Epoch')
plt.ylabel('Validation Accuracy')
plt.title('Transfer Learning: Head-only vs Full Fine-tune')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/cv01_transfer.png', dpi=80)
plt.close()
print('Saved: /tmp/cv01_transfer.png')


## Real-World Example 2: Manual Data Augmentation Pipeline

In [ ]:
# ---- RW2: Data augmentation without torchvision --------------------------
# Implements horizontal flip, random crop, and color jitter using numpy.
# Augmentation prevents overfitting by creating new training views.


def random_horizontal_flip(img, p=0.5):
    """Flip image left-right with probability p.

    Args:
        img: numpy array (C, H, W) in [0, 1]
        p: probability of flipping

    Returns:
        Augmented image (C, H, W)
    """
    if np.random.random() < p:
        return img[:, :, ::-1].copy()  # flip last axis (width)
    return img


def random_crop(img, crop_size, padding=4):
    """Pad then randomly crop back to original-ish size.

    This is the standard augmentation used in CIFAR training.
    Padding adds zeros around the border; the crop picks a random subwindow.
    """
    C, H, W = img.shape
    # Pad with zeros on all spatial sides
    padded = np.pad(img, ((0, 0), (padding, padding), (padding, padding)),
                   mode='constant')
    # Pick a random crop position
    top = np.random.randint(0, 2 * padding)
    left = np.random.randint(0, 2 * padding)
    return padded[:, top:top + crop_size, left:left + crop_size]


def color_jitter(img, brightness=0.2, contrast=0.2, saturation=0.1):
    """Apply random brightness and contrast changes.

    Brightness: multiply by (1 + delta), delta ~ U[-brightness, +brightness]
    Contrast: shift toward mean, then scale
    Clipped to [0, 1] to stay in valid range.
    """
    # Brightness jitter
    b_delta = 1.0 + np.random.uniform(-brightness, brightness)
    img = img * b_delta
    # Contrast jitter: scale about channel mean
    c_delta = 1.0 + np.random.uniform(-contrast, contrast)
    for c in range(img.shape[0]):
        mean_c = img[c].mean()
        img[c] = (img[c] - mean_c) * c_delta + mean_c
    return np.clip(img, 0.0, 1.0)


def augment(img_chw):
    """Apply full augmentation pipeline to a single image (C, H, W)."""
    img = img_chw.copy().astype(np.float32)
    img = random_horizontal_flip(img, p=0.5)
    img = random_crop(img, crop_size=img.shape[1], padding=4)
    img = color_jitter(img, brightness=0.2, contrast=0.2)
    return img


# Generate a synthetic image and show 8 augmented versions
# Use class 3 color pattern from the synthetic dataset above
base_img = (
    np.random.randn(3, 32, 32).astype(np.float32) * 0.3
    + np.array([0.3, 0.15, 0.5])[:, None, None]
)
base_img = np.clip(base_img, 0, 1)

fig, axes = plt.subplots(2, 5, figsize=(14, 5))
axes[0, 0].imshow(np.transpose(base_img, (1, 2, 0)))
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for idx in range(1, 10):
    aug = augment(base_img)
    row, col = divmod(idx, 5)
    axes[row, col].imshow(np.transpose(aug, (1, 2, 0)))
    axes[row, col].set_title(f'Augmented {idx}')
    axes[row, col].axis('off')

plt.suptitle('RW2: Data Augmentation Pipeline (flip + crop + jitter)')
plt.tight_layout()
plt.savefig('/tmp/cv01_augmentation.png', dpi=80)
plt.close()
print('Saved: /tmp/cv01_augmentation.png')
print('Each augmented view is a different crop + flip + brightness setting.')


## Real-World Example 3: Skip Connections — ResNet vs Plain CNN

## Comparison: How Residual Connections Stabilize Training

In [ ]:
# ---- RW3 + Comparison: Residual vs plain CNN training stability -----------
# Skip connections solve the 'degradation problem': without them, adding more
# layers hurts training accuracy because gradients vanish at depth.


class PlainCNN(nn.Module):
    """Plain CNN with same depth as SmallResNet but NO skip connections.

    Used to demonstrate that residual connections are not just about depth
    but about gradient flow and optimization landscape.
    """

    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBNReLU(3, 32),
            ConvBNReLU(32, 32),
            nn.MaxPool2d(2),
            ConvBNReLU(32, 64),
            ConvBNReLU(64, 64),
            nn.MaxPool2d(2),
            ConvBNReLU(64, 128),
            ConvBNReLU(128, 128),
            nn.MaxPool2d(2),
            ConvBNReLU(128, 256),
            ConvBNReLU(256, 256),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)).flatten(1))


def quick_train(ModelClass, epochs=25, lr=0.05, tag=''):
    """Train a model for given epochs and return per-epoch train loss."""
    m = ModelClass(num_classes=N_CLASSES).to(device)
    opt = optim.SGD(m.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        m.train()
        ep_loss = 0.0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = crit(m(X_b), y_b)
            loss.backward()
            # Gradient clipping prevents explosive gradients in deep nets
            torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            opt.step()
            ep_loss += loss.item()
        sched.step()
        losses.append(ep_loss / len(train_loader))
    # Final val accuracy
    m.eval()
    correct = 0
    with torch.no_grad():
        for X_v, y_v in val_loader:
            preds = m(X_v.to(device)).argmax(1)
            correct += (preds == y_v.to(device)).sum().item()
    val_acc = correct / N_VAL
    print(f'{tag}: final val_acc = {val_acc:.3f}')
    return losses


COMPARE_EPOCHS = 25
plain_losses = quick_train(PlainCNN, epochs=COMPARE_EPOCHS, tag='Plain CNN')
resnet_losses = quick_train(SmallResNet, epochs=COMPARE_EPOCHS, tag='ResNet (skip)')

# Plot training loss curves to show how skip connections stabilize training
plt.figure(figsize=(9, 4))
plt.plot(plain_losses, label='Plain CNN (no skip)', color='tomato')
plt.plot(resnet_losses, label='ResNet (with skip connections)', color='steelblue')
plt.xlabel('Epoch')
plt.ylabel('Training Loss')
plt.title('Comparison: Residual vs Plain CNN — Skip Connections Stabilize Training')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/tmp/cv01_comparison.png', dpi=80)
plt.close()
print('Saved: /tmp/cv01_comparison.png')
print('ResNet should converge faster and more stably due to gradient highway.')
